# L1 — Break a tiny chain (Week 2)

**What you will do.** Hash a receipt, mine three blocks, quietly change a
payment, watch the verifier fail, then re-mine. About **60–75 minutes**.

**Turn in.** A short PDF (Q1–Q5) plus three PNGs this notebook saves in
`lab_output/`: `L1-hash.png`, `L1-mine.png`, `L1-tamper.png`.

**Stuck?** Handbook → Session 3. Same words as class: hash, nonce,
`prev_hash`, verify.

Run **Cell → Run All**. If a cell errors, read the comment above it.
Do not paste someone else's notebook.



## Words we use today

Read this once. If a later cell uses the word, come back here.

| Word | In this class it means |
|---|---|
| **Hash** | Fingerprint of exact bytes. One letter change → new print. |
| **Nonce** | How many guesses until the hash looks rare enough. |
| **Difficulty** | How rare (here: how many leading zeros). Stay at 3. |
| **prev_hash** | This page names the last page’s fingerprint. |
| **Verify** | Zeros still there AND each page points at the real last page. |



## 0. Folders for your screenshots

This cell makes `lab_output/` next to the notebook. Re-run it if you
moved files. Nothing secret goes in here.



In [ ]:
from pathlib import Path
import hashlib
from dataclasses import dataclass

import matplotlib.pyplot as plt

OUT = Path("lab_output")
OUT.mkdir(exist_ok=True)
print("screenshots will go to", OUT.resolve())



## Q1 — A tamper-evident seal (4 pts)

Hash a short payment receipt. Change **one** character. Hash again.
The two fingerprints should look unrelated. That is the seal.



In [ ]:
def sha256(text: str) -> str:
    """SHA-256 hex digest of a UTF-8 string. Same input → same print."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


receipt = "Ava pays Ben 2 EAGE for 10 min algebra"
tweaked = "Ava pays Ben 3 EAGE for 10 min algebra"  # one digit changed

h0 = sha256(receipt)
h1 = sha256(tweaked)
print("original:", h0)
print("tweaked: ", h1)
print("same?", h0 == h1)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.axis("off")
ax.set_title("L1-hash: one character, new fingerprint")
ax.text(0.02, 0.62, "original\n" + h0, family="monospace", fontsize=9)
ax.text(0.02, 0.18, "one digit changed\n" + h1, family="monospace", fontsize=9)
fig.savefig(OUT / "L1-hash.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved", OUT / "L1-hash.png")



**Write in the PDF (Q1).** In finance language, what did you just show?
2–4 sentences. Hint: a stored hash is a seal on the *exact* bytes.



## Q2 — Mine three pages (4 pts)

A **nonce** is the guess counter. **Difficulty 3** means the hash must
start with `000`. Stay at 3 — this is a classroom toy, not Bitcoin.



In [ ]:
@dataclass
class Block:
    prev_hash: str
    txs: str
    nonce: int = 0

    def header(self) -> str:
        return f"{self.prev_hash}|{self.txs}|{self.nonce}"

    def hash(self) -> str:
        return sha256(self.header())


def mine(prev_hash: str, txs: str, difficulty: int = 3) -> Block:
    """Brute-force nonce until the hash starts with `difficulty` zeros."""
    prefix = "0" * difficulty
    nonce = 0
    while True:
        cand = Block(prev_hash, txs, nonce)
        if cand.hash().startswith(prefix):
            return cand
        nonce += 1


DIFFICULTY = 3
genesis = mine("0" * 64, "class opens the lab", DIFFICULTY)
b2 = mine(genesis.hash(), "Ava->Ben:2", DIFFICULTY)
b3 = mine(b2.hash(), "Ben->Cara:1", DIFFICULTY)
chain = [genesis, b2, b3]

print(f"{'i':>2}  {'nonce':>8}  hash")
for i, b in enumerate(chain):
    print(f"{i:2d}  {b.nonce:8d}  {b.hash()}")

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.axis("off")
ax.set_title("L1-mine: three blocks, difficulty 3")
lines = [f"{i}  nonce={b.nonce}  {b.hash()[:18]}…" for i, b in enumerate(chain)]
ax.text(0.02, 0.5, "\n".join(lines), family="monospace", fontsize=10, va="center")
fig.savefig(OUT / "L1-mine.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved", OUT / "L1-mine.png")



**Write in the PDF (Q2).** What is a nonce, what is difficulty, and when
does a block pass? At most 8 sentences. Use what the cell printed.



## Q3 — Quiet edit, no re-mine (4 pts)

Change Ava’s payment in **block 1**. Do **not** re-mine. The verifier
should fail: either the puzzle zeros are gone, or the next page still
points at the old fingerprint.



In [ ]:
def verify_chain(chain, difficulty: int) -> bool:
    """PoW zeros on every block, and each parent pointer is honest."""
    prefix = "0" * difficulty
    for i, block in enumerate(chain):
        if not block.hash().startswith(prefix):
            print("FAIL: block", i, "hash no longer meets difficulty")
            return False
        if i == 0:
            continue
        if block.prev_hash != chain[i - 1].hash():
            print("FAIL: block", i, "points at a ghost parent")
            return False
    print("chain looks clean")
    return True


print("before vandalism:", verify_chain(chain, DIFFICULTY))
chain[1].txs = "Ava->Ben:200"  # quiet edit — no new nonce
after = verify_chain(chain, DIFFICULTY)
print("after a quiet edit:", after)

fig, ax = plt.subplots(figsize=(8, 2.4))
ax.axis("off")
ax.set_title("L1-tamper: edit block 1, do not re-mine")
ax.text(
    0.02,
    0.5,
    f"before: True\nafter:  {after}\n(expected False — PoW and/or parent link)",
    family="monospace",
    fontsize=11,
    va="center",
)
fig.savefig(OUT / "L1-tamper.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved", OUT / "L1-tamper.png")



**Write in the PDF (Q3).** What fails first, and why? Paste the True/False
line plus two sentences.



## Q4 — Re-mine from the break (4 pts)

You must re-mine the tampered block **and every block after it**. On a
long public chain that cost is the security budget.



In [ ]:
# Rebuild from block 1 forward. Genesis stays. Tampered txs stay.
repaired = [chain[0]]
n_remined = 0
for old in chain[1:]:
    nxt = mine(repaired[-1].hash(), old.txs, DIFFICULTY)
    repaired.append(nxt)
    n_remined += 1

print("re-mined", n_remined, "blocks (tampered + everything after)")
print("verifies again?", verify_chain(repaired, DIFFICULTY))
print("new hashes:")
for i, b in enumerate(repaired):
    print(i, b.hash()[:20], "txs=", b.txs)



**Write in the PDF (Q4).** How many blocks did you re-mine? Why is that
expensive on a long public chain? Four sentences. Not a sermon about electricity.



## Q5 — Unbundle for *your* coin (4 pts)

Edit the two lists. Class ERC-20s sit on someone else’s chain.



In [ ]:
# Edit these. Pairs the key looks for: used vs not-needed.
used_here = ["hash", "nonce", "prev_hash", "verify"]
not_needed_to_issue_an_erc20 = [
    "P2P gossip",
    "an energy market",
    "native L1 issuance",
    "mining pools",
]
print("We used:", ", ".join(used_here))
print("We did not need, to issue a class ERC-20:", ", ".join(not_needed_to_issue_an_erc20))
print("PDF: pick TWO from each list and say why in a sentence each.")



## Checklist before you export the PDF

1. `lab_output/L1-hash.png`, `L1-mine.png`, `L1-tamper.png` exist.
2. Q1–Q5 are in **your** words (talking with a teammate is OK).
3. Optional extra: Naranjo *Proof of Work Example* if you want another
   implementation of the same verbs —
   https://www.lorenzonaranjo.com/class-materials/fintech/notebooks.html

